Generates intermediate LaTeX tables from repeated-trial evaluation summaries. These tables were used as the numerical source for the manuscript tables; final manuscript formatting and table organization were subsequently edited manually.


In [1]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "src").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not locate project root containing src/")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

SRC = PROJECT_ROOT / "src"
ARTIFACTS = PROJECT_ROOT / "artifacts"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CWD:", Path.cwd())
print("ARTIFACTS:", ARTIFACTS)

PROJECT_ROOT: .
CWD: .
ARTIFACTS: ./artifacts


In [2]:

def tex_num_pm(mean, std):
    return rf"\num{{{mean:.8e}}} $\pm$ \num{{{std:.8e}}}"


def get_metric(summary_df, metric):
    row = summary_df.loc[summary_df["metric"] == metric]
    if row.empty:
        return r"\num{} $\pm$ \num{}"
    row = row.iloc[0]
    return tex_num_pm(row["mean"], row["trial_std"])


def make_feature_comparison_table(
    nonparam_fit_csv,
    param_fit_csv,
    output_tex,
):
    nonparam = pd.read_csv(nonparam_fit_csv)
    param = pd.read_csv(param_fit_csv)

    rows = [
        (r"$E^{\mathrm{rel}}_{res}$", "E_res_relative"),
        (r"$E^{\mathrm{rel}}_{\phi}$", "E_phi_relative"),
        (r"$E^{\mathrm{rel}}_{f}$", "E_env_relative"),
    ]

    body = "\n".join(
        rf"""
{label}
& {get_metric(nonparam, metric)}
& {get_metric(param, metric)} \\
"""
        for label, metric in rows
    )

    tex = rf"""
\begin{{table}}[ht]
\centering
\caption{{
Comparison of relative residual and feature-recovery metrics for the fully non-parametric and semi-parametric estimation algorithms.
The table reports the mean error $\\pm$ one standard deviation over $T_r = 10$ independent learning trials.
}}
\label{{tab:feature_comparison}}
\begin{{tabular}}{{lcc}}
\toprule
Metric
& Fully non-parametric
& Semi-parametric \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\end{{table}}
"""

    Path(output_tex).write_text(tex)
    return tex

In [3]:
def get_traj_value(traj_df, scenario, kind):
    """
    kind = 'mean' for bar E, 'std' for sigma_E.
    """
    row = traj_df.loc[traj_df["scenario"] == scenario]
    if row.empty:
        return r"\num{} $\pm$ \num{}"

    row = row.iloc[0]

    if kind == "mean":
        return tex_num_pm(row["mean_traj_mean"], row["mean_traj_trial_std"])
    if kind == "std":
        return tex_num_pm(row["std_traj_mean"], row["std_traj_trial_std"])

    raise ValueError("kind must be 'mean' or 'std'.")


def make_trajectory_comparison_table(
    nonparam_traj_csv,
    param_traj_csv,
    output_tex,
):
    nonparam = pd.read_csv(nonparam_traj_csv)
    param = pd.read_csv(param_traj_csv)

    blocks = [
        (
            r"$M_{\mathrm{train}}$",
            "train full [t0,T]",
            "train full [T,Tf]",
        ),
        (
            r"$M_{\mathrm{test}}$",
            "test N [t0,T]",
            "test N [T,Tf]",
        ),
    ]

    body_parts = []

    for dataset, recon_scenario, pred_scenario in blocks:
        body_parts.append(
            rf"""
\multirow{{4}}{{*}}{{{dataset}}}
& $\bar E_{{\mathrm{{recon}}}}$
& {get_traj_value(nonparam, recon_scenario, "mean")}
& {get_traj_value(param, recon_scenario, "mean")} \\

& $\sigma_{{E_{{\mathrm{{recon}}}}}}$
& {get_traj_value(nonparam, recon_scenario, "std")}
& {get_traj_value(param, recon_scenario, "std")} \\

& $\bar E_{{\mathrm{{pred}}}}$
& {get_traj_value(nonparam, pred_scenario, "mean")}
& {get_traj_value(param, pred_scenario, "mean")} \\

& $\sigma_{{E_{{\mathrm{{pred}}}}}}$
& {get_traj_value(nonparam, pred_scenario, "std")}
& {get_traj_value(param, pred_scenario, "std")} \\
\midrule
"""
        )

    body = "\n".join(body_parts).rstrip("\\midrule\n")

    tex = rf"""
\begin{{table}}[ht]
\centering
\caption{{
Comparison of trajectory-level errors for the fully non-parametric and semi-parametric estimation algorithms.
The table reports mean $\pm$ standard deviation over 10 independent learning trials.
}}
\label{{tab:trajectory_comparison}}
\begin{{tabular}}{{llcc}}
\toprule
Dataset & Statistic
& Fully non-parametric
& Semi-parametric \\
\midrule
{body} \\
\bottomrule
\end{{tabular}}
\end{{table}}
"""

    Path(output_tex).write_text(tex)
    return tex

In [4]:
from ips_learning.visualization.utils import TableRepository


PUBLICATION_PROFILES = {
    "kuramoto": "profile1",
    "levi": "levi_velocity_grid",
    "spp": "spp_velocity_grid_clumps",
}


def create_tables(SYSTEM):
    if SYSTEM not in PUBLICATION_PROFILES:
        raise ValueError(
            f"No publication profile configured for system: {SYSTEM}"
        )

    PROFILE_NAME = PUBLICATION_PROFILES[SYSTEM]

    evaluation_dir = (
        ARTIFACTS
        / SYSTEM
        / "evaluations"
        / "repeated_trials"
    )

    table_repo = TableRepository(root_dir=ARTIFACTS)

    publication_dir = table_repo.stage_dir(
        system_name=SYSTEM,
        profile_name=PROFILE_NAME,
        stage_name="publication",
    )

    make_feature_comparison_table(
        nonparam_fit_csv=(
            evaluation_dir
            / "fit_diagnostic_summary.csv"
        ),
        param_fit_csv=(
            evaluation_dir
            / "parametric_f"
            / "fit_diagnostic_summary.csv"
        ),
        output_tex=(
            publication_dir
            / "feature_comparison_table.tex"
        ),
    )

    make_trajectory_comparison_table(
        nonparam_traj_csv=(
            evaluation_dir
            / "trajectory_summary.csv"
        ),
        param_traj_csv=(
            evaluation_dir
            / "parametric_f"
            / "trajectory_summary.csv"
        ),
        output_tex=(
            publication_dir
            / "trajectory_comparison_table.tex"
        ),
    )


# Kuramoto

In [5]:
SYSTEM = "kuramoto"
create_tables(SYSTEM)

# Levi

In [6]:
SYSTEM = "levi"
create_tables(SYSTEM)

# SPP

In [7]:
SYSTEM = "spp"
create_tables(SYSTEM)